In [47]:
import folium
import osmnx as ox
import networkx as nx

khach_hang = [
    ['Khach 1', 10.761208381640975, 106.66835646578188],
    ['Khach 2', 10.783304116512499, 106.69462528061219],
    ['Khach 3', 10.773376523822351, 106.67750683643312],
    ['Khach 4', 10.790579315471804, 106.69939015362725]
]

xe = [
    ['Xe A', 10.762757567108883, 106.67021457267154],
    ['Xe B', 10.772706477635213, 106.69803150174114],
    ['Xe C', 10.795312129720845, 106.72206380944814],
    ['Xe D', 10.777488766355205, 106.69521071211855]
]

trung_tam = [10.765, 106.675]

G = ox.graph_from_point(
    trung_tam,
    dist=8000,
    network_type='drive'
)

def tim_nut(diem):
    return ox.distance.nearest_nodes(G, diem[2], diem[1])

def lay_toa_do(duong):
    toa_do = []

    for nut in duong:
        vi_do = G.nodes[nut]['y']
        kinh_do = G.nodes[nut]['x']
        toa_do.append([vi_do, kinh_do])

    return toa_do

ket_qua = []
xe_chua_ghep = xe.copy()

for khach in khach_hang:
    nut_khach = tim_nut(khach)

    xe_gan_nhat = None
    duong_ngan_nhat = None
    do_dai_nho_nhat = 999999999

    for tai_xe in xe_chua_ghep:
        nut_xe = tim_nut(tai_xe)

        try:
            do_dai = nx.shortest_path_length(
                G,
                nut_xe,
                nut_khach,
                weight='length'
            )

            duong = nx.shortest_path(
                G,
                nut_xe,
                nut_khach,
                weight='length',
                method='dijkstra'
            )

            if do_dai < do_dai_nho_nhat:
                do_dai_nho_nhat = do_dai
                xe_gan_nhat = tai_xe
                duong_ngan_nhat = duong

        except:
            pass

    if xe_gan_nhat != None:
        ket_qua.append([
            khach[0],
            xe_gan_nhat[0],
            round(do_dai_nho_nhat / 1000, 2),
            duong_ngan_nhat
        ])

        xe_chua_ghep.remove(xe_gan_nhat)

for dong in ket_qua:
    print(dong[0], 'duoc ghep voi', dong[1], '- quang duong', dong[2], 'km')

m = folium.Map(location=trung_tam, zoom_start=12)

for khach in khach_hang:
    folium.Marker(
        [khach[1], khach[2]],
        popup=khach[0],
        tooltip='Khach hang',
        icon=folium.Icon(color='red')
    ).add_to(m)

for tai_xe in xe:
    folium.Marker(
        [tai_xe[1], tai_xe[2]],
        popup=tai_xe[0],
        tooltip='Tai xe',
        icon=folium.Icon(color='green')
    ).add_to(m)

for dong in ket_qua:
    ten_khach = dong[0]
    ten_xe = dong[1]
    do_dai = dong[2]
    duong = dong[3]

    folium.PolyLine(
        lay_toa_do(duong),
        color='blue',
        weight=4,
        popup=ten_xe + ' den ' + ten_khach + ': ' + str(do_dai) + ' km'
    ).add_to(m)

m


Khach 1 duoc ghep voi Xe A - quang duong 0.38 km
Khach 2 duoc ghep voi Xe D - quang duong 0.82 km
Khach 3 duoc ghep voi Xe B - quang duong 2.85 km
Khach 4 duoc ghep voi Xe C - quang duong 4.14 km
